# Лабораторная работа №13: Продвинутый RAG (Hybrid Search + Re-ranking)

**Студент:** [ФИО]  
**Группа:** [Номер группы]

## Цель работы
Изучить продвинутые техники улучшения RAG-систем:
- Hybrid Search (комбинация семантического и ключевого поиска)
- Re-ranking результатов для повышения релевантности
- Работа с библиотеками LangChain, ChromaDB, FlagEmbedding

## Теоретические сведения

### Hybrid Search
Гибридный поиск объединяет два подхода:
1. **Dense Retrieval** (семантический поиск) — использует векторные эмбеддинги для поиска по смыслу
2. **Sparse Retrieval** (ключевой поиск) — использует BM25 или TF-IDF для поиска по точным совпадениям слов

Преимущества гибридного подхода:
- Лучшее покрытие запросов разных типов
- Повышение recall за счет комбинации методов
- Устойчивость к проблемам каждого отдельного метода

### Re-ranking
Переранжирование — процесс повторной сортировки найденных документов с использованием более точной, но медленной модели.

Типичный пайплайн:
1. Быстрый поиск (hybrid) → 50-100 кандидатов
2. Cross-encoder модель → оценка релевантности каждого кандидата
3. Сортировка по score → топ-N документов для генерации ответа

Популярные модели для re-ranking:
- BGE-Reranker
- Cohere Rerank
- Jina Reranker

## Задание

### Базовый уровень
1. Установите необходимые библиотеки
2. Создайте коллекцию документов для тестирования
3. Реализуйте простой hybrid search (векторный + BM25)
4. Сравните результаты чистого векторного поиска и гибридного

### Продвинутый уровень
1. Интегрируйте cross-encoder модель для re-ranking
2. Реализуйте полный пайплайн: retrieval → re-rank → generation
3. Оцените качество с помощью метрик (precision@k, MRR)

### Индивидуальный уровень
1. Исследуйте влияние веса между dense и sparse поиском на качество
2. Сравните разные модели для re-ranking
3. Предложите оптимизации для продакшн-использования

## Ход работы

In [ ]:
# Установка необходимых библиотек
!pip install -q langchain langchain-community chromadb sentence-transformers rank_bm25 flag-embedding

In [ ]:
import numpy as np
from typing import List, Tuple, Dict
from langchain.docstore.document import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Шаг 1: Создание тестового датасета
documents_data = [
    "Машинное обучение — это подраздел искусственного интеллекта, который использует статистические методы для обучения компьютеров.",
    "Глубокое обучение использует нейронные сети с множеством слоев для обработки сложных паттернов в данных.",
    "Трансформеры — архитектура нейронных сетей, основанная на механизме внимания, революционизировавшая NLP.",
    "RAG (Retrieval-Augmented Generation) комбинирует поиск информации с генерацией текста языковыми моделями.",
    "Векторные базы данных хранят эмбеддинги и позволяют выполнять быстрый семантический поиск по сходству.",
    "BM25 — алгоритм ранжирования, используемый поисковыми системами для оценки релевантности документов запросу.",
    "Cross-encoder модели оценивают пару (запрос, документ) одновременно, обеспечивая высокую точность ранжирования.",
    "Fine-tuning — процесс дообучения предобученной модели на специфическом датасете для конкретной задачи.",
    "Prompt engineering — искусство составления эффективных инструкций для языковых моделей.",
    "Квантование моделей снижает размер и ускоряет инференс за счет уменьшения точности весов."
]

documents = [Document(page_content=text, metadata={"id": i}) for i, text in enumerate(documents_data)]
print(f"Создано {len(documents)} документов")

In [ ]:
# Шаг 2: Инициализация эмбеддинг модели и векторного хранилища
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma.from_documents(documents, embedding_model)
print("Векторное хранилище создано")

In [ ]:
# Шаг 3: Инициализация BM25 retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5
print("BM25 retriever настроен")

In [ ]:
# Шаг 4: Функция для гибридного поиска
def hybrid_search(query: str, vectorstore, bm25_retriever, top_k: int = 5, alpha: float = 0.5) -> List[Tuple[Document, float]]:
    """
    Гибридный поиск с взвешенной комбинацией результатов.
    alpha: вес векторного поиска (1-alpha — вес BM25)
    """
    # Векторный поиск
    vector_results = vectorstore.similarity_search_with_score(query, k=top_k * 2)
    
    # BM25 поиск
    bm25_results = bm25_retriever.invoke(query)
    
    # Нормализация скороров и объединение
    doc_scores = {}
    
    # Обработка векторных результатов (чем меньше score, тем лучше в Chroma)
    max_vector_score = max([score for _, score in vector_results]) if vector_results else 1
    min_vector_score = min([score for _, score in vector_results]) if vector_results else 0
    
    for doc, score in vector_results:
        # Инвертируем и нормализуем (превращаем расстояние в схожесть)
        normalized_score = 1 - (score / (max_vector_score + 0.001))
        doc_id = id(doc)
        if doc_id not in doc_scores:
            doc_scores[doc_id] = {"doc": doc, "vector_score": 0, "bm25_score": 0}
        doc_scores[doc_id]["vector_score"] = normalized_score
    
    # Обработка BM25 результатов
    for i, doc in enumerate(bm25_results[:top_k * 2]):
        doc_id = id(doc)
        if doc_id not in doc_scores:
            doc_scores[doc_id] = {"doc": doc, "vector_score": 0, "bm25_score": 0}
        # Простая нормализация по позиции
        doc_scores[doc_id]["bm25_score"] = 1 - (i / (len(bm25_results) + 1))
    
    # Комбинирование скороров
    combined_results = []
    for doc_id, data in doc_scores.items():
        final_score = alpha * data["vector_score"] + (1 - alpha) * data["bm25_score"]
        combined_results.append((data["doc"], final_score))
    
    # Сортировка по убыванию скорора
    combined_results.sort(key=lambda x: x[1], reverse=True)
    
    return combined_results[:top_k]

In [ ]:
# Шаг 5: Тестирование гибридного поиска
query = "Как работают нейронные сети в глубоком обучении?"

print(f"Запрос: {query}\n")

# Чистый векторный поиск
print("=== Векторный поиск ===")
vector_results = vectorstore.similarity_search_with_score(query, k=3)
for i, (doc, score) in enumerate(vector_results, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   Текст: {doc.page_content[:100]}...\n")

# Чистый BM25 поиск
print("=== BM25 поиск ===")
bm25_results = bm25_retriever.invoke(query)
for i, doc in enumerate(bm25_results[:3], 1):
    print(f"{i}. {doc.page_content[:100]}...\n")

# Гибридный поиск
print("=== Гибридный поиск (alpha=0.5) ===")
hybrid_results = hybrid_search(query, vectorstore, bm25_retriever, top_k=3, alpha=0.5)
for i, (doc, score) in enumerate(hybrid_results, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   Текст: {doc.page_content[:100]}...\n")

In [ ]:
# Шаг 6: Инициализация cross-encoder для re-ranking
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-encoder модель загружена")

In [ ]:
# Шаг 7: Функция для re-ranking
def rerank_documents(query: str, documents: List[Document], reranker, top_k: int = 3) -> List[Tuple[Document, float]]:
    """
    Переранжирование документов с помощью cross-encoder.
    """
    if not documents:
        return []
    
    # Подготовка пар (запрос, документ)
    pairs = [[query, doc.page_content] for doc in documents]
    
    # Получение скороров от cross-encoder
    scores = reranker.predict(pairs)
    
    # Комбинирование документов с скорами
    ranked_docs = list(zip(documents, scores))
    
    # Сортировка по убыванию скорора
    ranked_docs.sort(key=lambda x: x[1], reverse=True)
    
    return ranked_docs[:top_k]

In [ ]:
# Шаг 8: Полный пайплайн с re-ranking
query = "Что такое трансформеры и механизм внимания?"

print(f"Запрос: {query}\n")

# 1. Гибридный поиск для получения кандидатов
candidate_docs = hybrid_search(query, vectorstore, bm25_retriever, top_k=5, alpha=0.5)
candidates = [doc for doc, _ in candidate_docs]

print("=== Кандидаты после гибридного поиска ===")
for i, (doc, score) in enumerate(candidate_docs, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   {doc.page_content[:80]}...\n")

# 2. Re-ranking
reranked_docs = rerank_documents(query, candidates, reranker, top_k=3)

print("=== После re-ranking ===")
for i, (doc, score) in enumerate(reranked_docs, 1):
    print(f"{i}. Re-rank Score: {score:.4f}")
    print(f"   {doc.page_content[:80]}...\n")

In [ ]:
# Шаг 9: Оценка качества (упрощенная)
def calculate_metrics(results: List[Tuple[Document, float]], relevant_ids: List[int]) -> Dict[str, float]:
    """
    Расчет метрик Precision@k и MRR.
    relevant_ids: ID релевантных документов
    """
    k = len(results)
    
    # Precision@k
    relevant_retrieved = sum(1 for doc, _ in results if doc.metadata.get("id") in relevant_ids)
    precision_at_k = relevant_retrieved / k if k > 0 else 0
    
    # MRR (Mean Reciprocal Rank)
    mrr = 0.0
    for i, (doc, _) in enumerate(results, 1):
        if doc.metadata.get("id") in relevant_ids:
            mrr = 1.0 / i
            break
    
    return {
        "precision_at_k": precision_at_k,
        "mrr": mrr
    }

# Пример оценки (предполагаем, что релевантны документы с id 2 и 3)
relevant_doc_ids = [2, 3]
metrics = calculate_metrics(reranked_docs, relevant_doc_ids)

print("=== Метрики качества ===")
print(f"Precision@{len(reranked_docs)}: {metrics['precision_at_k']:.2f}")
print(f"MRR: {metrics['mrr']:.2f}")

## Контрольные вопросы

1. В чем преимущества гибридного поиска перед использованием только одного метода?
2. Как работает алгоритм BM25 и чем он отличается от TF-IDF?
3. Почему cross-encoder точнее bi-encoder для задачи ранжирования?
4. Какой параметр alpha оптимальный для гибридного поиска и как его подобрать?
5. Какие проблемы могут возникнуть при использовании re-ranking в продакшене?

## Выводы

В данной лабораторной работе были изучены продвинутые техники улучшения RAG-систем:
- Реализован гибридный поиск, сочетающий семантический и ключевой поиск
- Интегрирован cross-encoder для переранжирования результатов
- Построен полный пайплайн retrieval → re-rank → evaluation
- Проанализированы метрики качества поиска

Гибридный подход позволяет улучшить recall системы, а re-ranking повышает precision финальных результатов, что критически важно для качества ответов в RAG-приложениях.